## myPolar — Pretrained Model Fine-tuning (Task-1 / Task-2 / Task-3)

AI Engineering (Fundamental) Batch-2 — Group Project
ဒီ notebook က **pretrained multilingual model တွေကို fine-tune လုပ်ပြီး** myPolar ဒေတာရဲ့ Task သုံးခုမှာ နှိုင်းယှဉ်တဲ့ အပိုင်းပါ။

| Task | ဘာလုပ်လဲ | အမျိုးအစား | Metric |
|---|---|---|---|
| Task-1 | Polarized လား မဟုတ်ဘူးလား | binary | macro-F1 |
| Task-2 | Polarization type (political, racial/ethnic, religious, gender/sexual, other) | multi-label | macro-F1 + per-label F1 |
| Task-3 | Manifestation (stereotype, vilification, dehumanization, extreme_language, lack_of_empathy, invalidation) | multi-label | macro-F1 + per-label F1 |

Task တစ်ခုချင်းစီမှာ model ၄ ခု run ပါတယ်: `xlm-roberta-base`, `bert-base-multilingual-cased`, `microsoft/mdeberta-v3-base`, `NLLB-200 (distilled-600M) encoder`

References
- POLAR SemEval-2026 Task 9: https://polar-semeval.github.io/
- Task paper: https://arxiv.org/html/2604.06817v1
- POLAR benchmark paper: https://aclanthology.org/2026.findings-acl.1433.pdf

**GPU ပေါ်မှာ run ပါ** (Colab / Kaggle / lab server)။ CPU မှာ အရမ်းနှေးပါတယ်။

# Assignment 7 - Polarization Detection in Myanmar Text using Pretrained Multilingual Models

The three tasks are:

- **Task-1**: Polarization detection (binary)
- **Task-2**: Polarization type (5 labels, multi-label: political, racial/ethnic, religious, gender/sexual, other)
- **Task-3**: Polarization manifestation (6 labels, multi-label: stereotype, vilification, dehumanization, extreme language, lack of empathy, invalidation)

This notebook covers the **pretrained-model fine-tuning** part only.

The experiments and workflow include:

1. Environment setup and configuration
2. Data loading and cleaning (multi-line quoted records, duplicate and conflicting-label removal, non-Myanmar rows removal)
3. Train / validation / test splitting (80 / 10 / 10, stratified on `polarization`, seed 42)
4. Task-specific datasets (Task-2 and Task-3 use polarized rows only)
5. Fine-tuning function (HuggingFace `Trainer`, best epoch chosen by dev macro-F1)
6. Task-1 with 4 models: XLM-R base, mBERT, mDeBERTa-v3-base, NLLB-200-600M encoder
7. Task-2 with the same 4 models (BCE loss, threshold tuned on dev)
8. Task-3 with the same 4 models (BCE loss, threshold tuned on dev)
9. Model evaluation using test macro-F1 and per-label F1
10. Error analysis (confusion matrix, misclassified examples, rare label `religious`)
11. Result summary (model comparison)

## Data
- **Source**: Course repository, Assignment-2 folder: https://github.com/ye-kyaw-thu/AIE-F-B2/tree/main/assignment/assignment-2
- **Dataset**: myPolar, the Myanmar subset of the POLAR dataset (SemEval-2026 Task 9).
- **Files used**:
  - `task1_polarization_no_id.txt` (Task-1: polarization, binary)
  - `task2_polarization_type_no_id.txt` (Task-2: polarization type, 5 labels)
  - `task3_polarization_manifestation_no_id.txt` (Task-3: manifestation, 6 labels)
- **Format**: `|`-delimited text files. Some records contain multi-line quoted text, so a custom loader is used.
- **Cleaning in the notebook**: remove exact duplicates and texts with conflicting labels, drop non-Myanmar rows, merge the three task files by text.
- **Split**: 80 / 10 / 10 (train / dev / test), stratified on `polarization`, seed 42. Saved in `splits/`.

## Models
| Model | Checkpoint | Note |
|---|---|---|
| XLM-R base | `xlm-roberta-base` | multilingual encoder |
| mBERT | `bert-base-multilingual-cased` | multilingual encoder |
| mDeBERTa-v3-base | `microsoft/mdeberta-v3-base` | multilingual DeBERTa |
| NLLB encoder | `facebook/nllb-200-distilled-600M` | encoder only + mean pooling + linear head |

## Settings
`EPOCHS=5`, `LR=2e-5`, `BATCH=16`, `MAX_LEN=128`, `SEED=42`, fp16 on GPU (RTX 4060 8GB).

## Files
- `finetune_pretrained.ipynb` : main notebook
- `task1_polarization_no_id.txt`, `task2_polarization_type_no_id.txt`, `task3_polarization_manifestation_no_id.txt` : data
- `splits/` : train / dev / test CSVs
- `outputs_pretrained/results.csv` : all results
- `outputs_pretrained/*_test_predictions.csv` : test predictions

## How to run
GPU and internet (HuggingFace model download) are required.

## Limitations
- Single seed, small Task-2/3 test sets
- `MAX_LEN=128` truncates 14.8% of sentences for mBERT (3.4% for XLM-R)
- Own 80/10/10 split (the paper uses 70/10/20)
- No hyper-parameter search

##  Setup

In [1]:
import os, re, json, random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, set_seed)

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

torch 2.14.1+cu126 | cuda: True


## 1. Configuration

File names and training settings (epochs, batch size, etc.) are collected here. The models themselves are run one by one in the task sections (Section 6 onward).

- `DROP_INCONSISTENT = True`: drops rows with `polarization = 0` that still carry type/manifestation labels from Task-2/3 (inconsistent annotation). The number of dropped rows is reported in the data section.
- Do not change `SEED` or the split ratio. Use the shared split files (`splits/*.csv`) so that the results are comparable with the teammates' baselines.

In [ ]:
TASK1_FILE = "task1_polarization_no_id.txt"      
TASK2_FILE = "task2_polarization_type_no_id.txt"
TASK3_FILE = "task3_polarization_manifestation_no_id.txt"
OUT_DIR    = "outputs_pretrained"      
SPLIT_DIR  = "splits"                  

SEED = 42
DROP_INCONSISTENT = True               

EPOCHS, LR, BATCH, MAX_LEN = 5, 2e-5, 16, 128
USE_POS_WEIGHT = False                 

TYPE_COLS = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
MANI_COLS = ["stereotype", "vilification", "dehumanization",
             "extreme_language", "lack_of_empathy", "invalidation"]

os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(SPLIT_DIR, exist_ok=True)
set_seed(SEED)

## 2. Data: read, merge and clean the three task files

We use only the three group files (`text|label...` format, `|`-delimited).

`pd.read_csv` fails on these files because some texts contain **newlines** (one record spans several lines). A custom loader is used instead.

Cleaning steps
1. Read the files with the custom loader (records with newlines are re-joined; rows with malformed labels are skipped).
2. Merge Task-1 with Task-2/3 on the text, so that each Task-2/3 row also gets its `polarization` value.
3. Remove duplicate texts: for exact duplicates keep the first one, and drop texts whose labels conflict.
4. Remove rows that contain no Myanmar characters.

In [41]:
def load_task_file(path):
    """'text|l1|l2|...' ဖိုင်ကို DataFrame အဖြစ် ဖတ်။ စာသားထဲ newline ပါတဲ့ record တွေကိုပါ ပြန်ဆက်။"""
    L = open(path, encoding="utf-8-sig").read().split("\n")
    hdr = L[0].strip().split("|"); k = len(hdr) - 1               
    tail = r"[01]?(\|[01]?){%d}" % (k - 1)
    only_labels, ends_labels = re.compile("^" + tail + "$"), re.compile(r"\|" + tail + "$")
    rows, buf, skipped = [], None, []
    for l in L[1:]:
        if buf is None and not l.strip(): continue
        if buf is not None:                                        
            if only_labels.match(l):
                rows.append((" ".join(buf), l)); buf = None
            elif ends_labels.search(l):
                txt = l.rsplit("|", k)[0]; rows.append((" ".join(buf + [txt]), l[len(txt) + 1:])); buf = None
            else:
                buf.append(l)
            continue
        if ends_labels.search(l):
            txt = l.rsplit("|", k)[0]; rows.append((txt, l[len(txt) + 1:]))
        elif l.startswith('"'):                                   
            buf = [l]
        else:
            skipped.append(l[:60])                                 
    if skipped: print(f"{path}: no label skipped row {len(skipped)} ")
    assert buf is None, "No finish record " + str(buf)[:80]
    out = [[re.sub(r"\s+", " ", t).strip().strip('"').strip()] + b.split("|") for t, b in rows]
    d = pd.DataFrame(out, columns=hdr)
    d[hdr[1:]] = d[hdr[1:]].replace("", np.nan).apply(pd.to_numeric, errors="coerce")
    return d

t1, t2, t3 = load_task_file(TASK1_FILE), load_task_file(TASK2_FILE), load_task_file(TASK3_FILE)
print("rows  task1:", len(t1), "| task2:", len(t2), "| task3:", len(t3))
assert (t2["text"].values == t3["text"].values).all(), "Task-2 နဲ့ Task-3 row မကိုက်ဘူး"
assert list(t2.columns[1:]) == TYPE_COLS and list(t3.columns[1:]) == MANI_COLS, "column အမည် မကိုက်ဘူး"

task1_polarization_no_id.txt: no label skipped row 1 
rows  task1: 10311 | task2: 10313 | task3: 10313


In [ ]:

def dedup(d):
    label_cols = list(d.columns[1:])
    key = d[label_cols].fillna(-1).astype(int).astype(str).apply("|".join, axis=1)
    conflict = key.groupby(d["text"]).nunique()
    bad = set(conflict[conflict > 1].index)
    out = d[~d["text"].isin(bad)].drop_duplicates("text", keep="first")
    return out, len(bad), len(d) - len(bad) - len(out)

parts = {}
for name, d in [("task1", t1), ("task2", t2), ("task3", t3)]:
    parts[name], n_conf, n_dup = dedup(d)
    print(f"{name}: conflicting texts dropped {n_conf} | exact duplicates dropped {n_dup}")

df = parts["task1"].merge(parts["task2"], on="text", how="inner").merge(parts["task3"], on="text", how="inner")
df = df[df["polarization"].isin([0, 1])].copy()
df["polarization"] = df["polarization"].astype(int)

no_my = ~df["text"].str.contains(r"[\u1000-\u109F]")
print("no-Myanmar rows dropped:", int(no_my.sum()))
df = df[~no_my].reset_index(drop=True)
print("clean:", df.shape, "| polarized ratio:", round(df.polarization.mean(), 3))

task1: conflicting texts dropped 1 | exact duplicates dropped 50
task2: conflicting texts dropped 13 | exact duplicates dropped 50
task3: conflicting texts dropped 12 | exact duplicates dropped 50
no-Myanmar rows dropped: 19
clean: (10224, 13) | polarized ratio: 0.53


## 3. Train / Dev / Test split (80 / 10 / 10)

The split is done **once**, on the merged table, stratified on `polarization`. Task-2 and Task-3 are then filtered out of this same split, so there is no train/test leakage between tasks.

The split files are saved in `splits/`. 

In [6]:
idx = np.arange(len(df))
tr_i, tmp_i = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=df["polarization"])
dv_i, te_i  = train_test_split(tmp_i, test_size=0.5, random_state=SEED, stratify=df.loc[tmp_i, "polarization"])

keep = ["text", "polarization"] + TYPE_COLS + MANI_COLS
splits = {"train": df.loc[tr_i, keep], "dev": df.loc[dv_i, keep], "test": df.loc[te_i, keep]}
for k, v in splits.items():
    v.to_csv(f"{SPLIT_DIR}/{k}.csv", index=False)
    print(k, v.shape, "| polarized:", round(v.polarization.mean(), 3))

train (8179, 13) | polarized: 0.53
dev (1022, 13) | polarized: 0.529
test (1023, 13) | polarized: 0.53


## 4. Prepare the datasets for each task

- **Task-1**: all rows, label = `polarization`.
- **Task-2 / Task-3**: keep only rows with `polarization = 1`. Rows with missing labels (NaN) are removed, and every row must have at least one positive label.
- If `DROP_INCONSISTENT = True`, rows with `polarization = 0` that still carry type/manifestation labels are dropped.

In [7]:
def make_task(split_df, task):
    d = split_df.copy()
    if task == "task1":
        return d[["text"]].assign(**{"polarization": d["polarization"]}), ["polarization"]
    cols = TYPE_COLS if task == "task2" else MANI_COLS
    if DROP_INCONSISTENT:
        d = d[d["polarization"] == 1]
    d = d.dropna(subset=cols)
    d = d[d[cols].sum(axis=1) > 0]
    return d[["text"] + cols].astype({c: int for c in cols}), cols

summary = []
for t in ["task1", "task2", "task3"]:
    for name, sp in splits.items():
        d, cols = make_task(sp, t)
        summary.append({"task": t, "split": name, "rows": len(d),
                        **{c: int(d[c].sum()) for c in cols}})
pd.DataFrame(summary).fillna("")

,task,split,rows,polarization,political,racial/ethnic,religious,gender/sexual,other,stereotype,vilification,dehumanization,extreme_language,lack_of_empathy,invalidation
0,task1,train,8179,4332.0,,,,,,,,,,,
1,task1,dev,1022,541.0,,,,,,,,,,,
2,task1,test,1023,542.0,,,,,,,,,,,
3,task2,train,3802,,1875.0,634.0,297.0,521.0,1191.0,,,,,,
4,task2,dev,472,,235.0,73.0,36.0,61.0,147.0,,,,,,
5,task2,test,484,,225.0,88.0,30.0,59.0,161.0,,,,,,
6,task3,train,3942,,,,,,,1127.0,2650.0,817.0,1539.0,926.0,1304.0
7,task3,dev,484,,,,,,,117.0,314.0,101.0,193.0,123.0,166.0
8,task3,test,491,,,,,,,147.0,320.0,94.0,186.0,124.0,163.0


## 5. Fine-tune function

- Single label (Task-1) uses `CrossEntropy`. Multiple labels (Task-2/3) use `BCEWithLogits` (multi-label).
- The best epoch is selected by **dev macro-F1**.
- For multi-label tasks, the decision threshold is tuned on the dev set (0.1 to 0.9) instead of the default 0.5, and then applied to the test set.
- Test predictions are saved as CSV files for error analysis.

In [ ]:
def f1_macro(y, p, multi):
    return f1_score(y, p, average="macro", zero_division=0)

class WTrainer(Trainer):
    pos_weight = None
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        if labels.dtype == torch.float32:                      # multi-label
            pw = self.pos_weight.to(out.logits.device) if self.pos_weight is not None else None
            loss = torch.nn.functional.binary_cross_entropy_with_logits(out.logits, labels, pos_weight=pw)
        else:
            loss = torch.nn.functional.cross_entropy(out.logits, labels)
        return (loss, out) if return_outputs else loss

def run_one(model_name, task, epochs=EPOCHS, lr=LR, bs=BATCH, tag=None, model_fn=None):
    tag = tag or model_name.replace("/", "_")
    out = f"{OUT_DIR}/{task}_{tag}"
    tr, cols = make_task(splits["train"], task)
    dv, _ = make_task(splits["dev"], task)
    te, _ = make_task(splits["test"], task)
    multi = len(cols) > 1
    ydt = np.float32 if multi else np.int64
    tok = AutoTokenizer.from_pretrained(model_name)
    if "nllb" in model_name.lower(): tok.src_lang = "mya_Mymr"      # NLLB: မြန်မာစာ language token သုံး

    def enc(d):
        e = tok(d["text"].tolist(), truncation=True, max_length=MAX_LEN)
        return [{k: e[k][i] for k in e} for i in range(len(d))]
    class D(torch.utils.data.Dataset):
        def __init__(s, d):
            s.e = enc(d); s.y = d[cols].values.astype(ydt) if multi else d[cols[0]].values.astype(ydt)
        def __len__(s): return len(s.y)
        def __getitem__(s, i):
            x = dict(s.e[i]); x["labels"] = torch.tensor(s.y[i]); return x
    dtr, ddv, dte = D(tr), D(dv), D(te)

    def collate(batch):                      # input ကို pad၊ label ကို stack (multi-label ပါ ကိုက်)
        labels = torch.stack([b.pop("labels") for b in batch])
        out = tok.pad(batch, return_tensors="pt"); out["labels"] = labels
        return out

    n_out = len(cols) if multi else 2
    model = model_fn(n_out) if model_fn else AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=n_out,
        problem_type="multi_label_classification" if multi else "single_label_classification")
    model = model.float()          # weights ကို fp32 ပြောင်း (fp16 weights + fp16 training ကြောင့် error တက်တာ ပြင်)
            
    def metrics(p):
        lg, y = p
        pred = (1/(1+np.exp(-lg)) > 0.5).astype(int) if multi else lg.argmax(-1)
        return {"macro_f1": f1_macro(y, pred, multi)}

    args = TrainingArguments(
        output_dir=out, num_train_epochs=epochs, learning_rate=lr,
        per_device_train_batch_size=bs, per_device_eval_batch_size=bs*2,
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="macro_f1", weight_decay=0.01,
        warmup_steps=int(0.1 * epochs * -(-len(dtr) // bs)),   # 10% warmup (transformers 4.x/5.x ကိုက်)
       
        save_total_limit=1, report_to="none", seed=SEED, remove_unused_columns=False,
        fp16=torch.cuda.is_available(), logging_strategy="no")
    trn = WTrainer(model=model, args=args, train_dataset=dtr, eval_dataset=ddv,
                   data_collator=collate, compute_metrics=metrics)
    if multi and USE_POS_WEIGHT:
        pos = dtr.y.sum(0); trn.pos_weight = torch.tensor((len(dtr.y)-pos)/np.maximum(pos, 1), dtype=torch.float32)
    trn.train()

    # Tune the threshold on the dev set.
    thr = 0.5
    if multi:
        pd_ = trn.predict(ddv).predictions; sig = 1/(1+np.exp(-pd_))
        best = max(np.arange(0.1, 0.91, 0.05), key=lambda t: f1_macro(ddv.y, (sig > t).astype(int), True))
        thr = float(best)
    pt = trn.predict(dte).predictions
    if multi:
        sig = 1/(1+np.exp(-pt)); pred = (sig > thr).astype(int)
    else:
        pred = pt.argmax(-1)
    res = {"task": task, "model": model_name, "test_macro_f1": round(f1_macro(dte.y, pred, multi), 4),
           "dev_macro_f1": round(trn.evaluate()["eval_macro_f1"], 4), "threshold": thr,
           "n_train": len(dtr), "n_test": len(dte)}
    if multi:
        for j, c in enumerate(cols):
            res[f"f1_{c}"] = round(f1_score(dte.y[:, j], pred[:, j], zero_division=0), 4)
    # prediction saved
    pr = te.copy().reset_index(drop=True)
    for j, c in enumerate(cols if multi else ["polarization"]):
        pr[f"pred_{c}"] = pred[:, j] if multi else pred
    pr.to_csv(f"{out}_test_predictions.csv", index=False)
    del trn, model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return res

## Saving Results + run_experiments
This function runs a model across all 3 tasks in a single execution. 
Since it saves the output to results.csv after each task is done, it skips the already completed tasks and only runs the remaining ones.

In [ ]:
RES_CSV = f"{OUT_DIR}/results.csv"
results = pd.read_csv(RES_CSV).to_dict("records") if os.path.exists(RES_CSV) else []

def run_experiments(model_name, tasks=("task1", "task2", "task3"), tag=None, model_fn=None):
    for task in tasks:
        if any(r["task"] == task and r["model"] == model_name for r in results):
            print("Skip:", task, model_name, "(Already run)"); continue
        print(f"\n=== {task} | {model_name} ===")
        try:
            results.append(run_one(model_name, task, tag=tag, model_fn=model_fn))
        except Exception as e:                      
            print("FAILED:", model_name, task, repr(e)[:300]); continue
        pd.DataFrame(results).to_csv(RES_CSV, index=False)

## NLLB Encoder Classifier (Prep for Model 4)
Because NLLB is an encoder-decoder model, it doesn't work directly with AutoModelForSequenceClassification. We take just the encoder and add a mean-pooling layer plus a linear classification head.
Word embeddings are frozen to fit the 8GB GPU memory limit. This cell only contains the class definition (no output).

In [ ]:
from transformers import AutoModel
from transformers.modeling_outputs import SequenceClassifierOutput

class EncoderCls(torch.nn.Module):
    def __init__(self, name, n_out):
        super().__init__()
        full = AutoModel.from_pretrained(name)
        self.encoder = full.get_encoder() if hasattr(full, "get_encoder") else full
        for p in self.encoder.embed_tokens.parameters():      # Freeze word embeddings (the largest component) -> to fit in an 8GB GPU
            p.requires_grad = False
        self.drop = torch.nn.Dropout(0.1)
        self.head = torch.nn.Linear(self.encoder.config.d_model if hasattr(self.encoder.config, "d_model")
                                    else self.encoder.config.hidden_size, n_out)
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kw):   
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).float()
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
        return SequenceClassifierOutput(logits=self.head(self.drop(pooled)))

NL = "facebook/nllb-200-distilled-600M"

## 6. Task-1 — Polarization Detection — Polarized or Not (Binary)

We run the 4 models one by one on Task-1. After each run, the result is saved to `results.csv`, automatically skipping the ones that are already completed.

In [22]:
# Task-1 · Model 1 — XLM-RoBERTa
run_experiments("xlm-roberta-base", tasks=["task1"])

Skip: task1 xlm-roberta-base (Already run)


In [24]:
# Task-1 · Model 2 — Multilingual BERT
run_experiments("bert-base-multilingual-cased", tasks=["task1"])

Skip: task1 bert-base-multilingual-cased (Already run)


In [25]:
# Task-1 · Model 3 — mDeBERTa
run_experiments("microsoft/mdeberta-v3-base", tasks=["task1"])


=== task1 | microsoft/mdeberta-v3-base ===


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight   

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.693071,0.712276
2,No log,0.505149,0.785318
3,No log,0.545390,0.791580
4,No log,0.718008,0.804137
5,No log,0.763051,0.805636


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Macro F1
No log,0.763051,5,0.805636


In [26]:
# Task-1 · Model 4 — NLLB-200 encoder
run_experiments(NL, tasks=["task1"], tag="nllb_encoder", model_fn=lambda n_out: EncoderCls(NL, n_out))


=== task1 | facebook/nllb-200-distilled-600M ===


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

[transformers] M2M100Model LOAD REPORT from: facebook/nllb-200-distilled-600M
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.521345,0.741460
2,No log,0.509910,0.764057
3,No log,0.520818,0.773363
4,No log,0.562130,0.769757
5,No log,0.562295,0.779056


Training Loss,Validation Loss,Epoch,Macro F1
No log,0.562295,5,0.779056


## 7. Task-2 — Polarization Type — 5 Categories (Multi-label)

We run the 4 models sequentially on Task-2. Results are saved to `results.csv` after each run, automatically skipping previously completed models.

In [27]:
# Task-2 · Model 1 — XLM-RoBERTa
run_experiments("xlm-roberta-base", tasks=["task2"])


=== task2 | xlm-roberta-base ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.426332,0.243938
2,No log,0.329352,0.631389
3,No log,0.301649,0.701294
4,No log,0.290967,0.718166
5,No log,0.295842,0.716132


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Macro F1
No log,0.290967,5,0.718166


In [28]:
# Task-2 · Model 2 — Multilingual BERT
run_experiments("bert-base-multilingual-cased", tasks=["task2"])

Skip: task2 bert-base-multilingual-cased (Already run)


In [29]:
# Task-2 · Model 3 — mDeBERTa
run_experiments("microsoft/mdeberta-v3-base", tasks=["task2"])


=== task2 | microsoft/mdeberta-v3-base ===


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight   

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.370892,0.401946
2,No log,0.314635,0.603120
3,No log,0.282154,0.720154
4,No log,0.281238,0.747681
5,No log,0.282713,0.749797


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Macro F1
No log,0.282713,5,0.749797


In [30]:
# Task-2 · Model 4 — NLLB-200 encoder
run_experiments(NL, tasks=["task2"], tag="nllb_encoder", model_fn=lambda n_out: EncoderCls(NL, n_out))


=== task2 | facebook/nllb-200-distilled-600M ===


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

[transformers] M2M100Model LOAD REPORT from: facebook/nllb-200-distilled-600M
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.428151,0.201999
2,No log,0.355102,0.339693
3,No log,0.315582,0.620253
4,No log,0.303321,0.661264
5,No log,0.301182,0.677308


Training Loss,Validation Loss,Epoch,Macro F1
No log,0.301182,5,0.677308


## 8. Task-3 — Polarization Manifestation — 6 Types (Multi-label)

We run the 4 models sequentially on Task-3. Results are saved to `results.csv` after each run, automatically skipping previously completed models.

In [31]:
# Task-3 · Model 1 — XLM-RoBERTa
run_experiments("xlm-roberta-base", tasks=["task3"])


=== task3 | xlm-roberta-base ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.604760,0.131161
2,No log,0.561201,0.333547
3,No log,0.556231,0.435709
4,No log,0.545194,0.438087
5,No log,0.540495,0.479100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Macro F1
No log,0.540495,5,0.479100


In [32]:
# Task-3 · Model 2 — Multilingual BERT
run_experiments("bert-base-multilingual-cased", tasks=["task3"])

Skip: task3 bert-base-multilingual-cased (Already run)


In [33]:
# Task-3 · Model 3 — mDeBERTa
run_experiments("microsoft/mdeberta-v3-base", tasks=["task3"])


=== task3 | microsoft/mdeberta-v3-base ===


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight   

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.576408,0.132871
2,No log,0.540935,0.421279
3,No log,0.537269,0.494273
4,No log,0.542246,0.494993
5,No log,0.540508,0.528621


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Macro F1
No log,0.540508,5,0.528621


In [34]:
# Task-3 · Model 4 — NLLB-200 encoder
run_experiments(NL, tasks=["task3"], tag="nllb_encoder", model_fn=lambda n_out: EncoderCls(NL, n_out))


=== task3 | facebook/nllb-200-distilled-600M ===


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

[transformers] M2M100Model LOAD REPORT from: facebook/nllb-200-distilled-600M
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.590948,0.131161
2,No log,0.565747,0.221549
3,No log,0.552122,0.364368
4,No log,0.544262,0.408284
5,No log,0.542503,0.436060


Training Loss,Validation Loss,Epoch,Macro F1
No log,0.542503,5,0.436060


## 9. Result Summary

3 Tasks × 4 Models: **Test Set Macro-F1** (including per-label F1 scores for Task-2 and Task-3).

In [35]:
res_df = pd.DataFrame(results)
main = res_df.pivot(index="model", columns="task", values="test_macro_f1")
print("Test macro-F1"); display(main)
for t in ["task2", "task3"]:
    if t in set(res_df.task):
        cols = [c for c in res_df.columns if c.startswith("f1_")]
        sub = res_df[res_df.task == t].set_index("model")[cols].dropna(axis=1, how="all")
        print(f"\n{t} per-label F1"); display(sub)

Test macro-F1


task,task1,task2,task3
model,,,
bert-base-multilingual-cased,0.7730,0.6751,0.5787
facebook/nllb-200-distilled-600M,0.8130,0.6703,0.5700
microsoft/mdeberta-v3-base,0.8344,0.7288,0.5900
xlm-roberta-base,0.7876,0.7198,0.5659



task2 per-label F1


,f1_political,f1_racial/ethnic,f1_religious,f1_gender/sexual,f1_other
model,,,,,
bert-base-multilingual-cased,0.7689,0.6339,0.6557,0.6392,0.6777
xlm-roberta-base,0.7800,0.6587,0.7059,0.7692,0.6850
microsoft/mdeberta-v3-base,0.8018,0.6909,0.6667,0.7525,0.7323
facebook/nllb-200-distilled-600M,0.7727,0.5867,0.6349,0.6727,0.6847



task3 per-label F1


,f1_stereotype,f1_vilification,f1_dehumanization,f1_extreme_language,f1_lack_of_empathy,f1_invalidation
model,,,,,,
bert-base-multilingual-cased,0.4800,0.7825,0.5446,0.6089,0.4966,0.5594
xlm-roberta-base,0.5014,0.7891,0.5123,0.5891,0.4451,0.5581
microsoft/mdeberta-v3-base,0.5015,0.7945,0.5217,0.6169,0.5398,0.5657
facebook/nllb-200-distilled-600M,0.4772,0.7891,0.5455,0.5639,0.5055,0.5387


## 10. Error Analysis

(1) Generate a confusion matrix, and (2) Manually inspect misclassified sentences to identify error patterns (e.g., sarcasm, missing context, annotation errors, code-mixing, short texts).

In [36]:
TASK_E, MODEL_E = "task1", "xlm-roberta-base"
p = pd.read_csv(f"{OUT_DIR}/{TASK_E}_{MODEL_E.replace('/', '_')}_test_predictions.csv")
y, h = p["polarization"], p["pred_polarization"]
print(classification_report(y, h, target_names=["not polarized", "polarized"], digits=3))
print(pd.DataFrame(confusion_matrix(y, h), index=["true 0", "true 1"], columns=["pred 0", "pred 1"]))

wrong = p[y != h].copy()
wrong["len"] = wrong["text"].str.len()
print("\nmisclassified:", len(wrong), "| avg len wrong vs all:", round(wrong.len.mean()), round(p.text.str.len().mean()))
wrong.sample(min(15, len(wrong)), random_state=SEED)[["text", "polarization", "pred_polarization"]]

               precision    recall  f1-score   support

not polarized      0.804     0.732     0.766       481
    polarized      0.779     0.841     0.809       542

     accuracy                          0.790      1023
    macro avg      0.792     0.787     0.788      1023
 weighted avg      0.791     0.790     0.789      1023

        pred 0  pred 1
true 0     352     129
true 1      86     456

misclassified: 215 | avg len wrong vs all: 129 133


,text,polarization,pred_polarization
962,တော်လိုက်တဲ့ကလေးတွေ,0,1
1013,ထို့အပြင် စစ်ကော်မရှင်တပ်သည် ရနဲ ကျေးရွာအုပ်စု...,1,0
613,အဘဘောကို အကြီးကြီး မခဲ့တုန်းကပေါ့ကွယ်၊ အဘဘော လ...,1,0
837,အမှန်ပဲလက်တွေ့ကြုံးဖူးလိုသိတယ်ခွေးလောက်မှတန်ဖိ...,0,1
91,အိမ် တံခါး ပိတ် ၊ အွန်လိုင်း ပေါ် မှာ လူ တကာ လ...,1,0
490,တော်လှန်ရေးအဖွဲ့တချို့ကလည်း ဒါကိုနမူနာယူပြီး စ...,0,1
799,ထိုလူစား တို့ သည် မိမိ ၏ ချို့ယွင်းချက် ကို မမ...,0,1
326,ပြီးခဲ့ တဲ့ လ က Discord ဆော့ဖ်ဝဲလ် မှာ ၁၃ နှစ်...,0,1
980,မင်းပြောက်ဘုရင်၏ အာဏာစက်မှာ ရွှေနန်းတော်အတွင်း...,0,1
617,တစ် ကမ္ဘာ လုံး သောက် တလွဲ တွေ ဖြစ် နေ တာ,0,1


In [44]:
# Task-2/3: error analysis for one label
pd.set_option("display.max_colwidth", None)

def show_left(d):
    """Show the text column left-aligned."""
    display(d[["text"]].style
        .set_properties(subset=["text"], **{"text-align": "left", "white-space": "normal"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

TASK_E, MODEL_E, LABEL = "task2", "xlm-roberta-base", "religious"
f = f"{OUT_DIR}/{TASK_E}_{MODEL_E.replace('/', '_')}_test_predictions.csv"
if os.path.exists(f):
    p = pd.read_csv(f)
    fn = p[(p[LABEL] == 1) & (p[f"pred_{LABEL}"] == 0)]
    fp = p[(p[LABEL] == 0) & (p[f"pred_{LABEL}"] == 1)]
    print(f"{LABEL}: missed (FN)={len(fn)} | false alarm (FP)={len(fp)}")
    print("\n--- Missed (FN) ---");       show_left(fn.head(8))
    print("\n--- False alarm (FP) ---");  show_left(fp.head(8))

religious: missed (FN)=6 | false alarm (FP)=14

--- Missed (FN) ---


,text
84,ရခိုင်နဲ့ မြနိမာပြည်သူတွေကို ဆဲဖို့ မြန်မာစာသင်နေကြတာ ဖမ်းပါစေ ဒါကရခိုင်ပြည်လို ဆီလူးခံနေတာ မဟုတ်ဘူး ဖမ်းပါ အားပေးတယ်
186,အရင်လို ဂျင်းတွေထည့်လို့မရတော့ဘူး။ မဘသ လည်း စည်းကမ်းလိုက်နာရမယ် ဥပဒေ က ဆံရှည်တွေ ကတုံးတွေ နားမလည်ဘူး ကျောင်းနာမည်စတေကာ အကြီးကြီးတွေမှန်ရှေ့မှာတပ်ပြီး ဘယ်ကောင်မှ လူမထင်ဘူးဆိုတဲ့အပြုမူတွေရပ်သင့်ပြီ။
199,ငရဲ ကျတယ် ဆို တာ ပွက်ပွက် ဆူ နေ တဲ့ ဆီ အိုး ကြီး ထဲ ကိုယ် တုံး လုံး ချွတ် ပြီး ဆင်း ရ တာ
207,အရင်ဘဝ က ကောင် ကို အပြစ် ပုံချ ရ တာ နဲ့ နောက်ဘဝ က ကောင် အတွက် ပြင်ဆင် ရ မယ် ဘာညာ ဂျင်း နဲ့ လက်ရှိဘဝ က ကောင် က $ပေါ
276,Genocide နဲ့ ခွဲထွက်ခွင့်ကို ဆက်စပ်ထားပြီး အဆုံးမှာ ဂျီဟတ်စစ်ပွဲဆင်နွှဲလို့ ရတဲ့အထိ ပုံဖော်သွားမယ်လို့လည်း သိထားကြပါတယ်
288,ကတုံးအချင်းချင်းကျတော့ လိုက်ကာကွယ်နေလိုက်ကျတာ ပြည်သူတွေခံနေရတာ ဘာအသံမှမထွက်လာဘူး



--- False alarm (FP) ---


,text
89,MNDAA သည် အခြေအနေအပေါ် တုံ့ပြန်လှုပ်ရှားရသည့် သူပုန်အဖွဲ့အဆင့်မှသည် “ကိုးကန့်အထူးဒေသ (၁)” ဟု လူသိများသည့် စနစ်ကျသော ပြည်နယ်ပုံစံကဲ့သို့ အုပ်ချုပ်ရေးယန္တရားတစ်ခုအသွင်ဖြင့် ကူးပြောင်းလာခဲ့သည်။
117,သူ တို့ ကတော့ မြတ် လှပါ ဘိ နတ်ပြည် က ဆင်းလာ တဲ့ အတိုင်း ပဲ လှောင်ပြောင်ခြင်း ။
127,ထို့အပြင် ယင်းခေတ် ၌ နတ်သိကြား ကို လေးလေး နက်နက် ယုံကြည်ကြ သည် ဖြစ်လေရာ လူမင်း ဘုန်းကြီး နတ်မင်းမနေသာ ဘဲ သိကြားမင်း က ပင် ဤလူမျိုးခြား အား တမင်ဖန်တီး၍ ပို့လိုက်ခြင်း ဖြစ်သည် ဟု ယုံကြည်ပြောဆိုသမှု များ လည်း ပြုကြကုန် ၏ ။
150,ဗြိတိသျှအင်ပါယာကြီးကမှ တကယ့်နေကြီးကွ။
165,အချစ်ကို ဘယ်လောက် ကိုးကွယ်ကိုးကွယ် မချစ်သင့်တဲ့ လူတန်းစားဆိုတာ သပ်သပ်ရှိပါတယ်။
200,ဗမာလူမျိုးကြီးဝါဒ ကလည်း အမှန်တကယ်မရှိလို့လား။ ခေတ်အဆက်ဆက် နိုင်ငံခေါင်းဆောင်တွေ အားလုံးနီးပါး ဗမာတွေဘဲမဟုတ်လား။ ဗမာတွေ ကိုးကွယ်တဲ့ ဗုဒ္ဓဘာသာဘုန်းကြီးတွေ ခရစ်ယာန်နယ်မြေတွေမှာ စည်းရုံးခွင့်သာသနာပြု့ခွင့်ရတယ်။ အစိုးရကိုယ်တိုင် က အားပေးတယ်။ နိုင်ငံတော် ဘတ်ဂျက် နှင့် လစာ ပေးထားတယ်။ ခရစ်ယာန်ဘာသာ က လာ စည်းရုံသာသနာပြု့မယ် ဆိုရင် အတားအဆီတွေများစွာ။
248,ရာဇဝတ်ဘေး လွတ်အောင် ပြေးလာတာပေါ့ ငယ်လေးရာ၊ ဒီကျတော့လည်း မထူးဘူးဆိုပြီး ဂိုဏ်းဖွဲ့ ဓားပြတိုက်နေတာပေါ့။
269,တိုင်းရင်းသား တွေကတော့ ပုဂံဘုရား တွေကို ဗမာ ဘုရင်တွေ တည်ထားတာမို့ ဗုဒ္ဓဘာသာအချင်းချင်းတောင် သာသနာအတွက် မရှုစိမ့် နိုင်ဘူး သထုံသမိုင်း မြောက်ဦး သမိုင်းတွေနဲ့ နှစ်ပေါင်း တစ်ထောင်လောက်က သမိုင်းတွေကို အငြိုး အတေး တွေနဲ့ လူမျိုးသေး ဝါဒ တွေ စွဲနေကြတယ်။


In [ ]:
import re
MM = r"[\u1000-\u109F]"
def space_ratio(t): return t.count(" ") / max(len(t), 1)
d = pd.read_csv("splits/train.csv")          
print((d["text"].map(space_ratio) > 0.2).mean())   

0.11407262501528304


## Findings

**Test macro-F1 (single seed = 42, 80/10/10 split)**

| Model | Task-1 | Task-2 | Task-3 |
|---|---|---|---|
| mDeBERTa-v3-base | **0.834** | **0.729** | **0.590** |
| NLLB-200-600M (encoder) | 0.813 | 0.670 | 0.570 |
| XLM-R base | 0.788 | 0.720 | 0.566 |
| mBERT | 0.773 | 0.675 | 0.579 |

**1. Model comparison**
- mDeBERTa-v3-base is the best model on all three tasks.
- On Task-1, the NLLB encoder (0.813) is second, ahead of XLM-R (0.788) and mBERT (0.773). On Task-2/3 it does not beat XLM-R or mBERT, so translation pre-training seems to help most on the binary task.
- XLM-R is second on Task-2 (0.720) but last on Task-3 (0.566). A model that does well on one task is not guaranteed to do well on another.
- On Task-3 the four models lie between 0.566 and 0.590. With one seed and a small test set, this gap should not be read as a clear ranking.

**2. Task difficulty**
- F1 drops from Task-1 (binary) to Task-2 (5 labels) to Task-3 (6 labels).
- On Task-3 every model stays below 0.60 macro-F1. The six manifestation labels may overlap, and there are only about 3.9k training rows. 

**3. Rare labels**
- `religious` has only 30 test samples. XLM-R misses 6 (FN) and raises 14 false alarms (FP), so FP outnumber FN.
- F1 on labels with low support is unstable, because a single error changes it a lot.

**4. Task-1 error analysis (XLM-R)**
- 215 of 1,023 test rows are misclassified (accuracy 0.790).
- False positives (129) outnumber false negatives (86), so the model tends to over-predict "polarized".
- **Writing-style variation**: about 11.4% of the training texts are written with spaces between syllables. We checked whether this hurts the model. Accuracy was 0.789 for unspaced texts (n = 915) and 0.796 for spaced texts (n = 108), so there is no evidence that spacing style affects XLM-R on Task-1. No extra text normalization was applied. This check was done only for XLM-R on Task-1.

**5. Limitations**
- Single seed. Differences below about 0.01 to 0.02 should be treated as noise. The Task-2/3 test sets are small (about 480 rows).
- `MAX_LEN=128` truncates 14.8% of sentences for mBERT (3.4% for XLM-R), which may disadvantage mBERT.
- We use our own 80/10/10 split, so results are not directly comparable with the paper (70/10/20).
- Task-2/3 use only `polarization = 1` rows. About 1,916 rows with inconsistent labels were dropped.
- No hyper-parameter search (LR = 2e-5 and 5 epochs, fixed).
- The spacing check covers XLM-R on Task-1 only.

**6. Comparison with baselines**
